In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ["MPLCONFIGDIR"] = f"/tmp/{os.environ['USER']}_matplotlib_cache"

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [3]:
import sys
import warnings
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D
import seaborn as sns
import cogwheel.posterior
from cogwheel import utils, gw_utils, gw_plotting, plotting
import cogwheel.data

sys.path.insert(0, '/home/abbye.williams/GWPE/code')
import helpers

warnings.simplefilter("ignore", RuntimeWarning)

#### compute $\Delta\ln\mathcal L\equiv\ln\mathcal L_\mathrm{II}^\mathrm{max}(\psi) - \ln\mathcal L_\mathrm{I}^\mathrm{max}(\psi\pm\pi/4)$ & compare to $\ln\mathcal B$ from PE
We want
$$
\mathrm{max}_D\ln\mathcal L(d^U|\vec\theta) - \mathrm{max}_D\ln\mathcal L(d^L|\vec\theta')
$$
with $d^L$ the lensed data, $d^U$ the unlensed data, $\vec\theta$ the injected parameters, and $\vec\theta'=\vec\theta(\psi\pm\pi/4)$ the imposter parameters.

In [5]:
def compute_mismatch(post, ed_u, noise=False, plot=True, verbose=True):

    approximant = ed_u.injection['approximant']
    injected_par_dic = ed_u.injection['par_dic']

    if noise == False:
        # reinstantiate with zero noise
        ed_u_0 = ed_u.reinstantiate(strain=np.zeros_like(ed_u.strain))
        ed_u_0.inject_signal(par_dic=injected_par_dic, approximant=approximant)
    else:
        # otherwise just keep the event data the same
        ed_u_0 = ed_u

    # likelihood
    like_u_0 = post.likelihood
    like_u_0.event_data = ed_u_0
    like_u_0._set_summary()
    like_u_0.asd_drift = None

    # Type II lens
    ed_0 = ed_u_0.reinstantiate(strain=ed_u_0.strain * 1j)
    like_0 = deepcopy(like_u_0)
    like_0.event_data = ed_0
    like_0._set_summary()
    like_0.asd_drift = None

    # what's the likelihood of the unlensed event data given the injected parameters?
    maxlnlike_u = like_u_0.lnlike_fft(injected_par_dic)
    if verbose:
        print(f"max ln likelihood of the unlensed data given injected parameters is {maxlnlike_u:.1f}")

    # what's the likelihood of the lensed event data given the imposter parameters?
    shifts = [+np.pi/4, -np.pi/4]
    shift_strs = ['+ pi/4', '- pi/4']
    maxlnlikes = []
    for shift_str, shift in zip(shift_strs, shifts):
        maxlnlike_imposter = like_0.lnlike_fft(injected_par_dic | dict(psi=injected_par_dic['psi'] + shift))
        if verbose:
            print(f"max ln likelihood of the lensed data given unlensed imposter (psi -> psi {shift_str}) is {maxlnlike_imposter:.1f}")
        maxlnlikes.append(maxlnlike_imposter)
    # take the max
    idx_max = np.argmax(maxlnlikes)
    maxlnlike_imposter = maxlnlikes[idx_max]
    shift_best = shifts[idx_max]
    if verbose:
        print(f"taking psi {shift_strs[idx_max]} as the best imposter")

    # maximize the likelihood over distance given the unlensed imposter
    maxD_res = helpers.max_over_distance(like_0, injected_par_dic, shift_best)
    maxlnlike_imposter_dL = -maxD_res['fun']
    if verbose:
        print(f"max ln likelihood given imposter, maximized over distance, is {maxlnlike_imposter_dL:.1f} at {maxD_res['x']:.1f} Mpc")

    rhosq = np.sum(ed_0.injection['h_h'])
    maxlnlike_diff = maxlnlike_u - maxlnlike_imposter_dL
    lnlike_mismatch = (maxlnlike_u - maxlnlike_imposter_dL) / rhosq
    if verbose:
        print(f"max lnlike difference (mismatch) = {maxlnlike_diff:.1f} ({lnlike_mismatch:.3f})")

    if plot:
        shift_strs_latex = [r'$+ \pi/4$', r'$- \pi/4$']
        # compare the lensed and unlensed waveforms
        kwargs = dict(trng=(-.1, 0.), alpha=0.6, lw=3, data_plot_kwargs=dict(c='k', lw=1.5, ls='--', alpha=1.))
        like_0.plot_whitened_wf(injected_par_dic, c='royalblue', figsize=(10,6), zorder=100, plot_data=True, **kwargs)
        labels = ['Lensed data', 'Unlensed waveform (injected pars)']
        custom_lines = [Line2D([0], [0], **kwargs['data_plot_kwargs']),
                        Line2D([0], [0], c='royalblue', alpha=kwargs['alpha'], lw=kwargs['lw'])]
        # like.plot_whitened_wf(like.par_dic_0, c='seagreen', fig=plt.gcf(), **kwargs)
        c = 'crimson'
        for shift, shift_str in zip(shifts, shift_strs_latex):
            imposter_par_dic = injected_par_dic | dict(psi=injected_par_dic['psi'] + shift, d_luminosity=maxD_res['x'])
            alpha = 0.5 if shift == shift_best else 0.2
            lw = 3 if shift == shift_best else kwargs['lw']
            like_0.plot_whitened_wf(imposter_par_dic, c=c, fig=plt.gcf(), plot_data=False, **kwargs | dict(alpha=alpha, lw=lw))
            labels.append(r'Unlensed imposter ($\psi\rightarrow\psi$'f'{shift_str})')
            custom_lines.append(Line2D([0], [0], c=c, lw=kwargs['lw'], alpha=alpha))
        plt.gca().legend(custom_lines, labels, loc=4, fontsize=8)

    return maxlnlike_diff, rhosq

### load Halton injections

In [ ]:
# load injection parameters
approximant = 'IMRPhenomXPHM'

# get the SNR and compute the Bayes factor
resdir = f'/home/abbye.williams/GWPE/data/injections/Halton/{approximant}/IntrinsicLVCPrior'
seed = 12
nevents = 100
idx_existing = []
idx_to_run = []
n_resamples = 1000
lnBayes = {}
# parameters to collect
params = {}
for idx in range(nevents):
    # load this sample
    eventname = f'GW{seed}_{idx}'
    eventdir = os.path.join(resdir, eventname)
    try:
        event_data, samples, samples_dir = helpers.load_event_data_and_posterior_samples(eventdir, eventname)
        # add cos iota
        samples['cosiota'] = np.cos(samples['iota'])
        idx_existing.append(idx)
    except FileNotFoundError:
        print(f"no samples for idx {idx}. continuing")
        idx_to_run.append(idx)
        continue

    # add derived quantities to the injection dictionary
    injection_dict = event_data.get_init_dict()['injection']
    injected_par_dict = injection_dict['par_dic']
    helpers.add_derived_quantities_injection(injected_par_dict)
    # add cos iota
    injected_par_dict['cosiota'] = np.cos(injected_par_dict['iota'])

    # Bayes factor
    lnBayes_fn = os.path.join(samples_dir, 'lnBayes.npy')
    lnBayes[idx] = np.load(lnBayes_fn)

    # median from the posterior
    for par_id, injected_value in injected_par_dict.items():
        median_post = utils.quantile(samples[par_id], 0.5, weights=samples['weights'])
        try:
            params[par_id][idx] = (injected_value, median_post)
        except KeyError:    # if the dictionary doesn't exist yet
            params[par_id] = {}
            params[par_id][idx] = (injected_value, median_post)